In [0]:
df = spark.read.csv("/Volumes/workspace/default/ev_data/ev_sessions_raw.csv", header=True, inferSchema=True)

from pyspark.sql.window import Window
from pyspark.sql import functions as F

df_daily = df.groupBy("session_date").agg(F.count_distinct("session_id").alias("sessions"),
                                        F.round(F.sum("revenue_gbp"),2).alias("revenue"),
                                        F.round(F.sum("kwh_delivered"),2).alias("kwh")).orderBy("session_date") 
    

 
w = Window.orderBy("session_date").rowsBetween(-6,0)


df_daily = df_daily.withColumn("ma7",F.round(F.avg("revenue").over(w),2))
df_daily = df_daily.withColumn("prev_day",F.lag("revenue",1).over(Window.orderBy("session_date")))
df_daily = df_daily.withColumn("revnue_growth",F.round((F.col("revenue")-F.col("prev_day"))/F.col("prev_day"),2))
df_daily = df_daily.withColumn("revnue_weekly",F.round(F.lag("revenue",7).over(Window.orderBy("session_date")),2)) #其實係睇到星期一對上星一既數

df_daily.write.format("delta").mode("overwrite").saveAsTable("historical_metrcs_snapsshot") #就好似TIME TRAVEL 咁,可以睇返有咩改動
df_daily.write.format("delta").mode("append").save("/Volumes/workspace/default/ev_data/historical_metrcs_snapsshot")  #加數據
display(df_daily)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


session_date,sessions,revenue,kwh,ma7,prev_day,revnue_growth,revnue_weekly
2026-03-01,869,7699.4,15798.55,7699.4,null,null,null
2026-03-02,1159,9953.65,20560.69,8826.53,7699.4,0.29,null
2026-03-03,1050,9081.5,18788.47,8911.52,9953.65,-0.09,null
2026-03-04,1118,9858.1,20228.14,9148.16,9081.5,0.09,null
2026-03-05,1227,10411.06,21546.37,9400.74,9858.1,0.06,null
2026-03-06,1038,9060.24,18560.64,9343.99,10411.06,-0.13,null
2026-03-07,835,7254.44,14858.87,9045.48,9060.24,-0.2,null
2026-03-08,834,7160.88,14858.18,8968.55,7254.44,-0.01,7699.4
2026-03-09,997,8804.85,18063.86,8804.44,7160.88,0.23,9953.65
2026-03-10,1026,8780.79,18059.24,8761.48,8804.85,0.0,9081.5


In [0]:
%sql
SELECT * FROM historical_metrcs_snapsshot LIMIT 10;

DESCRIBE HISTORY historical_metrcs_snapsshot